In [3]:
from course_utils.paths import get_project_root, get_data_dir
import pandas as pd
print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

프로젝트 루트 : C:\dev\kant-axagent-study\llm-data-analysis-course
데이터 폴더 : C:\dev\kant-axagent-study\llm-data-analysis-course\data


In [7]:
RAW_DIR = get_project_root() / "data" / "raw"
PROCESSED_DIR = get_project_root() / "data" / "processed"
REPORT_DIR = get_project_root() / "reports"

In [8]:
import pandas as pd
import numpy as np

customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")

In [ ]:
# orders 데이터 타입 변환, 결측 확인
orders["order_date"] = pd.to_datetime(orders["order_date"], errors = "coerce")
orders.dtypes

order_id                  int64
customer_id               int64
order_date        datetime64[s]
payment_method              str
order_status                str
dtype: object

In [16]:
orders.head()

,order_id,customer_id,order_date,payment_method,order_status
0,1,123,NaT,card,completed
1,2,77,NaT,naver_pay,cancelled
2,3,138,NaT,bank_transfer,cancelled
3,4,57,NaT,kakao_pay,cancelled
4,5,125,NaT,card,cancelled


In [17]:
# 키 결측 확인
print(customers["customer_id"].isna().sum())
print(order_items["order_item_id"].isna().sum())
print(orders["order_id"].isna().sum())
print(products["product_id"].isna().sum())


0
0
0
0


In [19]:
# 키 중복 여부 확인
for frame, key in [(customers, "customer_id"), (order_items, "order_item_id"), (orders, "order_id"), (products, "product_id")]:
    print(key, "결측 갯수: ", frame[key].isna().sum())

customer_id 결측 갯수:  0
order_item_id 결측 갯수:  0
order_id 결측 갯수:  0
product_id 결측 갯수:  0


In [25]:
# 파생컬럼 total_price(unit_price * quantity)를 order_items에 추가
order_items["total_price"] = order_items["unit_price"] * order_items["quantity"]

In [26]:
print(customers["city"].value_counts)
print(customers["gender"].value_counts)
print(customers["age"].describe())

<bound method IndexOpsMixin.value_counts of 0      광주
1      대구
2      성남
3      울산
4      부산
       ..
145    성남
146    부산
147    고양
148    부산
149    대전
Name: city, Length: 150, dtype: str>
<bound method IndexOpsMixin.value_counts of 0      F
1      F
2      F
3      F
4      F
      ..
145    M
146    M
147    M
148    M
149    M
Name: gender, Length: 150, dtype: str>
count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64


미변량 EDA

In [30]:
# 상품 카데고리의 개수와 가격 통계
products["category"].value_counts(dropna = False)

category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [32]:
product_count = products["category"].value_counts(dropna = False).to_frame()
print(type(product_count))
product_count

<class 'pandas.DataFrame'>


,count
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7


In [33]:
products["price"].describe()

count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [ ]:
# 상품 카테고리별 분포
category_price = products.groupby("category", dropna = False).agg(
    product_count = ("product_id", "size"),
    mean_price = ("price", "mean"),
    median_price = ("price", "median")
)

print(category_price)

          product_count     mean_price  median_price
category                                            
도서                   14  106857.142857      118500.0
뷰티                   16  117687.500000      134000.0
생활용품                 16   96437.500000       89000.0
스포츠                  19  111578.947368      103000.0
식품                    7  137142.857143      147000.0
전자기기                 17  101588.235294      111000.0
패션                   11  115909.090909      115000.0


In [42]:
# 완료된 주문 병합 후 검증
completed_orders = orders[orders["order_status"] == "completed"]
print(completed_orders)
print(completed_orders["order_status"].value_counts())

     order_id  customer_id order_date payment_method order_status
0           1          123        NaT           card    completed
5           6           87        NaT      naver_pay    completed
8           9          145        NaT           card    completed
10         11           47        NaT           card    completed
11         12           98        NaT      kakao_pay    completed
..        ...          ...        ...            ...          ...
293       294          147        NaT  bank_transfer    completed
294       295          122        NaT  bank_transfer    completed
295       296          116        NaT           card    completed
296       297           22        NaT  bank_transfer    completed
298       299          135        NaT  bank_transfer    completed

[184 rows x 5 columns]
order_status
completed    184
Name: count, dtype: int64


In [49]:
# order_items에서 completed인 항목 뽑기
items_with_orders = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [50]:
comopleted_item_with_orders = items_with_orders[items_with_orders["order_status"] == "completed"]
print(comopleted_item_with_orders["order_status"].value_counts())

order_status
completed    474
Name: count, dtype: int64


In [54]:
# 1. 주문 정보 붙이기
completed_item_with_orders = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# 2. completed만 남기기
completed_item_with_orders = completed_item_with_orders[
    completed_item_with_orders["order_status"] == "completed"
]

# 3. 고객별 완료 주문 건수
customer_completed_orders = (
    completed_item_with_orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

print(customer_completed_orders)

customer_id
3      2
4      1
5      2
6      2
7      1
      ..
145    3
146    1
147    2
148    1
149    1
Name: order_id, Length: 100, dtype: int64


In [56]:
# 고객별 완료 주문 건수
customer_completed_orders = completed_item_with_orders.groupby("customer_id")["order_id"].nunique()
print(customer_completed_orders)

customer_id
3      2
4      1
5      2
6      2
7      1
      ..
145    3
146    1
147    2
148    1
149    1
Name: order_id, Length: 100, dtype: int64


In [ ]:
# 카테고리별 수량 합산하기, 카테고리별 완료 주문 금액을 합산

# 카테고리(products), 주문상태 완료(orders), 금액(order_items)

# 1 completed_items (orders, order_items)

# 2 category_items (completed_items, products)

In [ ]:
# 카테고리별 수량 합산하기, 카테고리별 완료 주문 금액 합산하기

# 1. completed_items 만들기 (orders + order_items)
completed_items = order_items.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# completed 주문만 남기기
completed_items = completed_items[
    completed_items["order_status"] == "completed"
]


# 2. category_items 만들기 (completed_items + products)
category_items = completed_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)


# 3. 카테고리별 수량 합계 + 완료 주문 금액 합계
category_summary = category_items.groupby(
    "category",
    dropna=False
).agg(
    total_quantity=("quantity", "sum"),
    total_sales=("total_price", "sum")
)

print(category_summary)